# Tarea 3: creación de ETLs con PySpark
Diana Paola Rojas Ramirez - 202116178

In [2]:
db_user = 'DB_202613_dp_rojasr1'
db_psswd = '202116178'
source_db_connection_string = 'jdbc:mysql://157.253.236.120:8080/WWImportersTransactional'
dest_db_connection_string = 'jdbc:mysql://157.253.236.120:8080/DB_202613_dp_rojasr1'
path_jar_driver = '/workspaces/ETL/drivers/mysql-connector-j.jar'

In [3]:
import os 
from pyspark.sql import functions as f, SparkSession, types as t
from pyspark import SparkContext, SparkConf, SQLContext
from pyspark.sql.functions import udf, col, length, isnan, when, count, regexp_replace
from datetime import datetime

In [4]:
#Configuración de la sesión
conf=SparkConf() \
    .set('spark.driver.extraClassPath', path_jar_driver)

spark_context = SparkContext(conf=conf)
sql_context = SQLContext(spark_context)
spark = sql_context.sparkSession

Picked up JAVA_TOOL_OPTIONS: -Xss512k -XX:+UseContainerSupport
Picked up JAVA_TOOL_OPTIONS: -Xss512k -XX:+UseContainerSupport
26/07/01 01:26:43 WARN Utils: Your hostname, codespaces-22a369 resolves to a loopback address: 127.0.0.1; using 10.0.10.14 instead (on interface eth0)
26/07/01 01:26:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/01 01:26:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
/home/vscode/.local/lib/python3.11/site-packages/pyspark/sql/context.py:113: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [5]:
def obterner_dataframe_desde_csv(_PATH, _sep):
    return spark.read.load(_PATH, format="csv", sep=_sep, inferSchema="true", header='true')

def obtener_dataframe_de_bd(db_connection_string, sql, db_user, db_psswd):
    df_bd = spark.read.format('jdbc')\
        .option('url', db_connection_string) \
        .option('dbtable', sql) \
        .option('user', db_user) \
        .option('password', db_psswd) \
        .option('driver', 'com.mysql.cj.jdbc.Driver') \
        .load()
    return df_bd

def guardar_db(db_connection_string, df, tabla, db_user, db_psswd):
    df.select('*').write.format('jdbc') \
      .mode('append') \
      .option('url', db_connection_string) \
      .option('dbtable', tabla) \
      .option('user', db_user) \
      .option('password', db_psswd) \
      .option('driver', 'com.mysql.cj.jdbc.Driver') \
      .save()

In [6]:
print("Existe driver:", os.path.exists(path_jar_driver))
print("Spark listo:", spark.version)

Existe driver: True
Spark listo: 3.5.1


In [7]:
test_fuente = obtener_dataframe_de_bd(
    source_db_connection_string,
    "(SELECT 1 AS prueba_fuente) AS t",
    db_user,
    db_psswd
)

test_destino = obtener_dataframe_de_bd(
    dest_db_connection_string,
    "(SELECT 1 AS prueba_destino) AS t",
    db_user,
    db_psswd
)

test_fuente.show()
test_destino.show()

+-------------+
|prueba_fuente|
+-------------+
|            1|
+-------------+

+--------------+
|prueba_destino|
+--------------+
|             1|
+--------------+



## Entregable 2 - Implementación del ETL

En esta sección se implementa el proceso ETL para el modelo multidimensional de movimientos de inventario. Se extraen datos desde la base transaccional WWImportersTransactional, se aplican las reglas de transformación definidas por el negocio y se cargan las tablas resultantes en la bodega de datos asignada.

Las dimensiones implementadas son Proveedor, TipoTransaccion y Fecha. Las dimensiones Producto y Cliente se reutilizan como dimensiones compartidas ya existentes en la bodega. Finalmente, se construye la tabla de hechos Hecho_Movimiento relacionando cada movimiento con las llaves sustitutas de sus dimensiones.

In [8]:
# EXTRACCIÓN - Dimensión Proveedor

sql_proveedores = '''(
SELECT DISTINCT
    ProveedorID AS ID_Proveedor_T,
    NombreProveedor AS Nombre,
    CategoriaProveedorID,
    PersonaContactoPrincipalID AS Contacto_principal,
    DiasPago AS Dias_pago,
    CodigoPostal AS Codigo_postal
FROM WWImportersTransactional.proveedores
) AS Temp_proveedores'''

sql_categorias_proveedores = '''(
SELECT DISTINCT
    CategoriaProveedorID,
    CategoriaProveedor AS Categoria
FROM WWImportersTransactional.CategoriasProveedores
) AS Temp_categorias_proveedores'''

proveedores = obtener_dataframe_de_bd(
    source_db_connection_string,
    sql_proveedores,
    db_user,
    db_psswd
)

categorias_proveedores = obtener_dataframe_de_bd(
    source_db_connection_string,
    sql_categorias_proveedores,
    db_user,
    db_psswd
)

proveedores.show(5)
categorias_proveedores.show(5)

+--------------+--------------------+--------------------+------------------+---------+-------------+
|ID_Proveedor_T|              Nombre|CategoriaProveedorID|Contacto_principal|Dias_pago|Codigo_postal|
+--------------+--------------------+--------------------+------------------+---------+-------------+
|             4|      Fabrikam, Inc.|                   4|                27|       30|        40351|
|             5|Graphic Design In...|                   2|                29|       14|        64847|
|             7|       Litware, Inc.|                   5|                33|       30|        95245|
|             9|      Nod Publishers|                   2|                37|        7|        27906|
|            10|Northwind Electri...|                   3|                39|       30|         7860|
+--------------+--------------------+--------------------+------------------+---------+-------------+
only showing top 5 rows

+--------------------+-------------------+
|CategoriaProv

In [9]:
# TRANSFORMACIÓN - Dimensión Proveedor

proveedores = proveedores.dropDuplicates()

proveedores = proveedores.join(
    categorias_proveedores,
    how="left",
    on="CategoriaProveedorID"
)

proveedores = proveedores.withColumn(
    "Dias_pago",
    f.when(f.col("Dias_pago") < 0, f.col("Dias_pago") * -1)
     .otherwise(f.col("Dias_pago"))
)

proveedores = proveedores.withColumn(
    "Nombre",
    regexp_replace(f.col("Nombre"), r"\s+(Inc|Ltd)\.?$", "")
)

proveedores = proveedores.dropDuplicates(["Nombre"])

proveedores = proveedores.coalesce(1).withColumn(
    "ID_Proveedor_DWH",
    f.monotonically_increasing_id() + 1
)

proveedores = proveedores.select(
    "ID_Proveedor_DWH",
    "ID_Proveedor_T",
    "Nombre",
    "Categoria",
    "Contacto_principal",
    "Dias_pago",
    "Codigo_postal"
)

proveedores.show(10)
proveedores.printSchema()

+----------------+--------------+--------------------+--------------------+------------------+---------+-------------+
|ID_Proveedor_DWH|ID_Proveedor_T|              Nombre|           Categoria|Contacto_principal|Dias_pago|Codigo_postal|
+----------------+--------------+--------------------+--------------------+------------------+---------+-------------+
|               1|             1| A Datum Corporation| productos novedosos|                21|       14|        46077|
|               2|             3|Consolidated Mess...|servicios de mens...|                25|       30|        94101|
|               3|             2|            Contoso,| productos novedosos|                23|        7|        98253|
|               4|             4|           Fabrikam,|                ropa|                27|       30|        40351|
|               5|             5|Graphic Design In...| productos novedosos|                29|       14|        64847|
|               6|             6| Humongous Insu

In [10]:
proveedores = proveedores.withColumn(
    "Nombre",
    regexp_replace(f.col("Nombre"), r"[,\s]+$", "")
)

proveedores.show(10)

+----------------+--------------+--------------------+--------------------+------------------+---------+-------------+
|ID_Proveedor_DWH|ID_Proveedor_T|              Nombre|           Categoria|Contacto_principal|Dias_pago|Codigo_postal|
+----------------+--------------+--------------------+--------------------+------------------+---------+-------------+
|               1|             1| A Datum Corporation| productos novedosos|                21|       14|        46077|
|               2|             3|Consolidated Mess...|servicios de mens...|                25|       30|        94101|
|               3|             2|             Contoso| productos novedosos|                23|        7|        98253|
|               4|             4|            Fabrikam|                ropa|                27|       30|        40351|
|               5|             5|Graphic Design In...| productos novedosos|                29|       14|        64847|
|               6|             6| Humongous Insu

In [10]:
# CARGUE - Dimensión Proveedor

guardar_db(dest_db_connection_string, proveedores, 'Proveedor', db_user, db_psswd)

In [11]:
#Validacion Proveedor
validar_proveedor = obtener_dataframe_de_bd(
    dest_db_connection_string,
    "(SELECT COUNT(*) AS total_proveedores FROM Proveedor) AS t",
    db_user,
    db_psswd
)

validar_proveedor.show()

+-----------------+
|total_proveedores|
+-----------------+
|               13|
+-----------------+



In [12]:
# EXTRACCIÓN - Dimensión TipoTransaccion

sql_tipos_transaccion = '''(
SELECT DISTINCT
    TipoTransaccionID AS ID_Tipo_Transaccion_T,
    TipoTransaccionNombre AS Tipo
FROM WWImportersTransactional.TiposTransaccion
) AS Temp_tipos_transaccion'''

tipo_transaccion = obtener_dataframe_de_bd(
    source_db_connection_string,
    sql_tipos_transaccion,
    db_user,
    db_psswd
)

tipo_transaccion.show(10)
tipo_transaccion.printSchema()

+---------------------+--------------------+
|ID_Tipo_Transaccion_T|                Tipo|
+---------------------+--------------------+
|                    2|Customer Credit Note|
|                    3|Customer Payment ...|
|                    4|     Customer Refund|
|                    5|    Supplier Invoice|
|                    6|Supplier Credit Note|
|                    7|Supplier Payment ...|
|                    8|     Supplier Refund|
|                    9|      Stock Transfer|
|                   10|         Stock Issue|
|                   11|       Stock Receipt|
+---------------------+--------------------+
only showing top 10 rows

root
 |-- ID_Tipo_Transaccion_T: integer (nullable = true)
 |-- Tipo: string (nullable = true)



In [13]:
# TRANSFORMACIÓN - Dimensión TipoTransaccion

tipo_transaccion = tipo_transaccion.dropDuplicates()

tipo_transaccion = tipo_transaccion.coalesce(1).withColumn(
    "ID_Tipo_Transaccion_DWH",
    f.monotonically_increasing_id() + 1
)

tipo_transaccion = tipo_transaccion.select(
    "ID_Tipo_Transaccion_DWH",
    "ID_Tipo_Transaccion_T",
    "Tipo"
)

tipo_transaccion.show(10)
tipo_transaccion.printSchema()

+-----------------------+---------------------+--------------------+
|ID_Tipo_Transaccion_DWH|ID_Tipo_Transaccion_T|                Tipo|
+-----------------------+---------------------+--------------------+
|                      1|                   10|         Stock Issue|
|                      2|                   12|Stock Adjustment ...|
|                      3|                   13|     Customer Contra|
|                      4|                    9|      Stock Transfer|
|                      5|                    4|     Customer Refund|
|                      6|                    2|Customer Credit Note|
|                      7|                    5|    Supplier Invoice|
|                      8|                    6|Supplier Credit Note|
|                      9|                    8|     Supplier Refund|
|                     10|                    3|Customer Payment ...|
+-----------------------+---------------------+--------------------+
only showing top 10 rows

root
 |-

In [14]:
# CARGUE - Dimensión TipoTransaccion

guardar_db(dest_db_connection_string, tipo_transaccion, 'TipoTransaccion', db_user, db_psswd)

In [14]:
validar_tipo_transaccion = obtener_dataframe_de_bd(
    dest_db_connection_string,
    "(SELECT COUNT(*) AS total_tipos_transaccion FROM TipoTransaccion) AS t",
    db_user,
    db_psswd
)

validar_tipo_transaccion.show()

+-----------------------+
|total_tipos_transaccion|
+-----------------------+
|                     12|
+-----------------------+



In [15]:
# EXTRACCIÓN - Dimensión Fecha

sql_fechas = '''(
SELECT DISTINCT
    FechaTransaccion
FROM WWImportersTransactional.movimientos
WHERE FechaTransaccion IS NOT NULL
) AS Temp_fechas'''

fechas = obtener_dataframe_de_bd(
    source_db_connection_string,
    sql_fechas,
    db_user,
    db_psswd
)

fechas.show(10)
fechas.printSchema()

+----------------+
|FechaTransaccion|
+----------------+
|     Jan 20,2014|
|     Jan 28,2014|
|     Feb 01,2014|
|     Mar 25,2014|
|     May 01,2014|
|     May 02,2014|
|     May 10,2014|
|     May 26,2014|
|     Jun 02,2014|
|     Jul 08,2014|
+----------------+
only showing top 10 rows

root
 |-- FechaTransaccion: string (nullable = true)



In [16]:
# TRANSFORMACIÓN - Dimensión Fecha

fechas = fechas.dropDuplicates()

fechas = fechas.withColumn(
    "Fecha",
    f.to_date(f.col("FechaTransaccion"))
)

fechas = fechas.filter(f.col("Fecha").isNotNull())

fechas = fechas.dropDuplicates(["Fecha"])

fechas = fechas.coalesce(1).withColumn(
    "ID_Fecha",
    f.monotonically_increasing_id() + 1
)

fechas = fechas.select(
    "ID_Fecha",
    "Fecha",
    f.dayofmonth("Fecha").alias("Dia"),
    f.month("Fecha").alias("Mes"),
    f.year("Fecha").alias("Anio"),
    f.weekofyear("Fecha").alias("Numero_semana_ISO")
)

fechas.show(10)
fechas.printSchema()

+--------+----------+---+---+----+-----------------+
|ID_Fecha|     Fecha|Dia|Mes|Anio|Numero_semana_ISO|
+--------+----------+---+---+----+-----------------+
|       1|2015-05-19| 19|  5|2015|               21|
|       2|2013-09-09|  9|  9|2013|               37|
|       3|2013-05-21| 21|  5|2013|               21|
|       4|2016-03-01|  1|  3|2016|                9|
|       5|2013-03-26| 26|  3|2013|               13|
|       6|2014-09-26| 26|  9|2014|               39|
|       7|2015-03-09|  9|  3|2015|               11|
|       8|2014-11-12| 12| 11|2014|               46|
|       9|2013-01-22| 22|  1|2013|                4|
|      10|2013-09-19| 19|  9|2013|               38|
+--------+----------+---+---+----+-----------------+
only showing top 10 rows

root
 |-- ID_Fecha: long (nullable = false)
 |-- Fecha: date (nullable = true)
 |-- Dia: integer (nullable = true)
 |-- Mes: integer (nullable = true)
 |-- Anio: integer (nullable = true)
 |-- Numero_semana_ISO: integer (nullable = 

In [18]:
# CARGUE - Dimensión Fecha

guardar_db(dest_db_connection_string, fechas, 'Fecha', db_user, db_psswd)

In [17]:
validar_fecha = obtener_dataframe_de_bd(
    dest_db_connection_string,
    "(SELECT COUNT(*) AS total_fechas FROM Fecha) AS t",
    db_user,
    db_psswd
)

validar_fecha.show()

+------------+
|total_fechas|
+------------+
|        1069|
+------------+



In [18]:
# EXTRACCIÓN - Tabla de hechos Hecho_Movimiento

sql_movimientos = '''(
SELECT DISTINCT
    TransaccionProductoID,
    ProductoID,
    TipoTransaccionID,
    ClienteID,
    ProveedorID,
    FechaTransaccion,
    Cantidad
FROM WWImportersTransactional.movimientos
WHERE FechaTransaccion IS NOT NULL
) AS Temp_movimientos'''

movimientos = obtener_dataframe_de_bd(
    source_db_connection_string,
    sql_movimientos,
    db_user,
    db_psswd
)

movimientos.show(10)
movimientos.printSchema()

+---------------------+----------+-----------------+---------+-----------+----------------+--------+
|TransaccionProductoID|ProductoID|TipoTransaccionID|ClienteID|ProveedorID|FechaTransaccion|Cantidad|
+---------------------+----------+-----------------+---------+-----------+----------------+--------+
|                94344|       108|               10|    185.0|       NULL|     Jan 20,2014|   -10.0|
|                96548|       162|               11|      0.0|        4.0|     Jan 28,2014|    10.0|
|                96560|       216|               10|    474.0|       NULL|     Jan 28,2014|   -10.0|
|                96568|        22|               11|      0.0|        7.0|     Jan 28,2014|    10.0|
|                96648|        25|               11|      0.0|        7.0|     Jan 28,2014|    10.0|
|                97822|        14|               10|    444.0|       NULL|     Feb 01,2014|   -10.0|
|                97832|        75|               11|      0.0|        7.0|     Feb 01,2014|

In [19]:
# LECTURA DE DIMENSIONES DESDE LA BODEGA

dim_proveedor = obtener_dataframe_de_bd(
    dest_db_connection_string,
    "(SELECT ID_Proveedor_DWH, ID_Proveedor_T FROM Proveedor) AS t",
    db_user,
    db_psswd
)

dim_tipo_transaccion = obtener_dataframe_de_bd(
    dest_db_connection_string,
    "(SELECT ID_Tipo_Transaccion_DWH, ID_Tipo_Transaccion_T FROM TipoTransaccion) AS t",
    db_user,
    db_psswd
)

dim_fecha = obtener_dataframe_de_bd(
    dest_db_connection_string,
    "(SELECT ID_Fecha, Fecha FROM Fecha) AS t",
    db_user,
    db_psswd
)

dim_producto = obtener_dataframe_de_bd(
    dest_db_connection_string,
    "(SELECT ID_Producto FROM Producto) AS t",
    db_user,
    db_psswd
)

dim_cliente = obtener_dataframe_de_bd(
    dest_db_connection_string,
    "(SELECT ID_Cliente_DWH, ID_Cliente_T FROM Cliente) AS t",
    db_user,
    db_psswd
)

In [20]:
print("Proveedor:", dim_proveedor.count())
print("TipoTransaccion:", dim_tipo_transaccion.count())
print("Fecha:", dim_fecha.count())
print("Producto:", dim_producto.count())
print("Cliente:", dim_cliente.count())

Proveedor: 13
TipoTransaccion: 12
Fecha: 1069
Producto: 227
Cliente: 664


In [21]:
movimientos.select("FechaTransaccion").distinct().show(30, False)

+---------------------------+
|FechaTransaccion           |
+---------------------------+
|Feb 27,2014                |
|2014-11-28 12:00:00.0000000|
|2014-06-26 12:00:00.0000000|
|2014-11-19 07:00:00.0000000|
|2015-02-18 07:00:00.0000000|
|2015-10-27 07:00:00.0000000|
|2016-01-27 07:00:00.0000000|
|2014-08-28 07:00:00.0000000|
|2014-09-19 07:00:00.0000000|
|2015-02-12 07:00:00.0000000|
|Jan 03,2015                |
|Sep 21,2015                |
|Aug 22,2015                |
|May 14,2016                |
|Jul 26,2014                |
|2014-04-18 12:00:00.0000000|
|2013-08-16 12:00:00.0000000|
|2013-04-26 07:00:00.0000000|
|2013-11-21 07:00:00.0000000|
|2015-01-16 07:00:00.0000000|
|Mar 25,2015                |
|Jul 16,2014                |
|Oct 14,2014                |
|2014-11-20 12:00:00.0000000|
|2015-11-30 12:00:00.0000000|
|2013-04-12 12:00:00.0000000|
|2013-02-21 07:00:00.0000000|
|2013-03-06 07:00:00.0000000|
|2014-09-24 07:00:00.0000000|
|Jul 18,2014                |
+---------

In [22]:
movimientos_test_fecha = movimientos.withColumn(
    "Fecha",
    f.coalesce(
        f.to_date(f.trim(f.col("FechaTransaccion")), "MMM dd,yyyy"),
        f.to_date(f.trim(f.col("FechaTransaccion")), "MMM dd, yyyy"),
        f.to_date(f.trim(f.col("FechaTransaccion")), "yyyy-MM-dd"),
        f.to_date(f.trim(f.col("FechaTransaccion")), "yyyy-MM-dd HH:mm:ss")
    )
)

movimientos_test_fecha.select("FechaTransaccion", "Fecha").show(30, False)

+----------------+----------+
|FechaTransaccion|Fecha     |
+----------------+----------+
|Jan 20,2014     |2014-01-20|
|Jan 28,2014     |2014-01-28|
|Jan 28,2014     |2014-01-28|
|Jan 28,2014     |2014-01-28|
|Jan 28,2014     |2014-01-28|
|Feb 01,2014     |2014-02-01|
|Feb 01,2014     |2014-02-01|
|Mar 25,2014     |2014-03-25|
|Mar 25,2014     |2014-03-25|
|Mar 25,2014     |2014-03-25|
|Mar 25,2014     |2014-03-25|
|Mar 25,2014     |2014-03-25|
|Mar 25,2014     |2014-03-25|
|Mar 25,2014     |2014-03-25|
|Mar 25,2014     |2014-03-25|
|Mar 25,2014     |2014-03-25|
|Mar 25,2014     |2014-03-25|
|Mar 25,2014     |2014-03-25|
|Mar 25,2014     |2014-03-25|
|Mar 25,2014     |2014-03-25|
|Mar 25,2014     |2014-03-25|
|Mar 25,2014     |2014-03-25|
|Mar 25,2014     |2014-03-25|
|May 01,2014     |2014-05-01|
|May 01,2014     |2014-05-01|
|May 01,2014     |2014-05-01|
|May 01,2014     |2014-05-01|
|May 02,2014     |2014-05-02|
|May 02,2014     |2014-05-02|
|May 02,2014     |2014-05-02|
+---------

In [23]:
# TRANSFORMACIÓN - Tabla de hechos Hecho_Movimiento

movimientos = movimientos.dropDuplicates()

movimientos = movimientos.withColumn(
    "Fecha",
    f.coalesce(
        f.to_date(f.trim(f.col("FechaTransaccion")), "MMM dd,yyyy"),
        f.to_date(f.trim(f.col("FechaTransaccion")), "MMM dd, yyyy"),
        f.to_date(f.substring(f.trim(f.col("FechaTransaccion")), 1, 10), "yyyy-MM-dd")
    )
)

movimientos = movimientos.filter(f.col("Fecha").isNotNull())

movimientos = movimientos.withColumn("ProductoID", f.col("ProductoID").cast("long")) \
    .withColumn("ClienteID", f.col("ClienteID").cast("int")) \
    .withColumn("ProveedorID", f.col("ProveedorID").cast("int")) \
    .withColumn("TipoTransaccionID", f.col("TipoTransaccionID").cast("int"))

hecho_movimiento = movimientos.join(dim_fecha, how="left", on="Fecha")

hecho_movimiento = hecho_movimiento.join(
    dim_producto,
    hecho_movimiento.ProductoID == dim_producto.ID_Producto,
    how="left"
)

hecho_movimiento = hecho_movimiento.join(
    dim_cliente,
    hecho_movimiento.ClienteID == dim_cliente.ID_Cliente_T.cast("int"),
    how="left"
)

hecho_movimiento = hecho_movimiento.join(
    dim_proveedor,
    hecho_movimiento.ProveedorID == dim_proveedor.ID_Proveedor_T,
    how="left"
)

hecho_movimiento = hecho_movimiento.join(
    dim_tipo_transaccion,
    hecho_movimiento.TipoTransaccionID == dim_tipo_transaccion.ID_Tipo_Transaccion_T,
    how="left"
)

hecho_movimiento = hecho_movimiento.select(
    "ID_Fecha",
    f.col("ID_Producto").alias("ID_Producto_DWH"),
    "ID_Proveedor_DWH",
    "ID_Cliente_DWH",
    "ID_Tipo_Transaccion_DWH",
    "Cantidad"
)

hecho_movimiento.show(10)
hecho_movimiento.printSchema()

+--------+---------------+----------------+--------------+-----------------------+--------+
|ID_Fecha|ID_Producto_DWH|ID_Proveedor_DWH|ID_Cliente_DWH|ID_Tipo_Transaccion_DWH|Cantidad|
+--------+---------------+----------------+--------------+-----------------------+--------+
|     976|             18|               7|          NULL|                     12|    10.0|
|     897|            155|               7|          NULL|                     12|    10.0|
|     667|            108|               7|          NULL|                     12|    10.0|
|     936|            135|               4|          NULL|                     12|    10.0|
|    1007|             68|               7|          NULL|                     12|    10.0|
|     695|            133|               4|          NULL|                     12|    10.0|
|     127|            216|               4|          NULL|                     12|    10.0|
|      95|            127|               7|          NULL|                     1

In [24]:
hecho_movimiento.select(
    f.count("*").alias("total_registros"),
    f.count(f.when(f.col("ID_Fecha").isNull(), 1)).alias("nulos_fecha"),
    f.count(f.when(f.col("ID_Producto_DWH").isNull(), 1)).alias("nulos_producto"),
    f.count(f.when(f.col("ID_Tipo_Transaccion_DWH").isNull(), 1)).alias("nulos_tipo_transaccion"),
    f.count(f.when(f.col("Cantidad").isNull(), 1)).alias("nulos_cantidad")
).show()

+---------------+-----------+--------------+----------------------+--------------+
|total_registros|nulos_fecha|nulos_producto|nulos_tipo_transaccion|nulos_cantidad|
+---------------+-----------+--------------+----------------------+--------------+
|         236667|          1|             0|                     0|             0|
+---------------+-----------+--------------+----------------------+--------------+



In [25]:
hecho_movimiento = hecho_movimiento.filter(f.col("ID_Fecha").isNotNull())

In [26]:
hecho_movimiento.select(
    f.count("*").alias("total_registros"),
    f.count(f.when(f.col("ID_Fecha").isNull(), 1)).alias("nulos_fecha"),
    f.count(f.when(f.col("ID_Producto_DWH").isNull(), 1)).alias("nulos_producto"),
    f.count(f.when(f.col("ID_Tipo_Transaccion_DWH").isNull(), 1)).alias("nulos_tipo_transaccion"),
    f.count(f.when(f.col("Cantidad").isNull(), 1)).alias("nulos_cantidad")
).show()

+---------------+-----------+--------------+----------------------+--------------+
|total_registros|nulos_fecha|nulos_producto|nulos_tipo_transaccion|nulos_cantidad|
+---------------+-----------+--------------+----------------------+--------------+
|         236666|          0|             0|                     0|             0|
+---------------+-----------+--------------+----------------------+--------------+



In [30]:
# CARGUE - Tabla de hechos Hecho_Movimiento

guardar_db(dest_db_connection_string, hecho_movimiento, 'Hecho_Movimiento', db_user, db_psswd)


In [27]:
hecho_movimiento.count()

236666